# Data preparation

Run this notebook once before opening a figure notebook. The cells below resolve the stable Zenodo concept record, verify the archive and its manifests, reject unsafe ZIP paths, and install the paper data atomically. The local archive fallback is for author testing only.


In [1]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import tempfile
import urllib.request
import zipfile

import pandas as pd
from IPython.display import Markdown, display

CONCEPT_RECORD_ID = "19241151"
ARCHIVE_NAME = "LinkD_Figure_Reproduction_Data.zip"
CHECKSUM_NAME = ARCHIVE_NAME + ".sha256"
ZENODO_API = f"https://zenodo.org/api/records/{CONCEPT_RECORD_ID}"
FORCE_REINSTALL = False
ALLOW_LOCAL_AUTHOR_FALLBACK = True

_candidates = []
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidates.extend((_base, _base / "For_Reviewer"))
ROOT = next((_p.resolve() for _p in _candidates if (_p / "notebooks").is_dir() and (_p / "README.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Could not locate the For_Reviewer folder.")
REPO = ROOT.parent
print("Reviewer package:", ROOT)


Reviewer package: /Users/cheng.wang/Documents/LinkD_Agent/For_Reviewer


## 1. Resolve the archive

Use the latest Zenodo version when it contains the reproduction bundle.


In [2]:
def fetch_json(url):
    request = urllib.request.Request(url, headers={"User-Agent": "LinkD-reviewer-reproduction/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.load(response)

record = fetch_json(ZENODO_API)
remote_files = {item["key"]: item for item in record.get("files", [])}
using_local_fallback = not {ARCHIVE_NAME, CHECKSUM_NAME}.issubset(remote_files)
if using_local_fallback:
    if not ALLOW_LOCAL_AUTHOR_FALLBACK:
        raise RuntimeError("The live Zenodo version does not yet contain the figure-reproduction archive.")
    archive_source = REPO / "zenodo_upload" / ARCHIVE_NAME
    checksum_source = REPO / "zenodo_upload" / CHECKSUM_NAME
    if not archive_source.is_file() or not checksum_source.is_file():
        raise FileNotFoundError("Zenodo has not published the bundle and the local author archive is unavailable.")
    display(Markdown("**AUTHOR-TESTING FALLBACK:** the live Zenodo record does not yet expose the new bundle. Do not release to reviewers until a new Zenodo version is published."))
else:
    archive_source = remote_files[ARCHIVE_NAME]["links"]["self"]
    checksum_source = remote_files[CHECKSUM_NAME]["links"]["self"]
    print("Resolved Zenodo record:", record["id"])
    print("Version DOI:", record.get("doi"))


**AUTHOR-TESTING FALLBACK:** the live Zenodo record does not yet expose the new bundle. Do not release to reviewers until a new Zenodo version is published.

## 2. Download and verify

The companion SHA-256, Zenodo checksum, safe-path rules, and internal bundle manifest are checked before extraction.


In [3]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_size):
            digest.update(block)
    return digest.hexdigest()

def copy_or_download(source, destination):
    if isinstance(source, Path):
        shutil.copy2(source, destination)
    else:
        request = urllib.request.Request(source, headers={"User-Agent": "LinkD-reviewer-reproduction/1.0"})
        with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as handle:
            shutil.copyfileobj(response, handle)

temporary_download = Path(tempfile.mkdtemp(prefix="linkd-reviewer-download-"))
downloaded_archive = temporary_download / ARCHIVE_NAME
downloaded_checksum = temporary_download / CHECKSUM_NAME
copy_or_download(archive_source, downloaded_archive)
copy_or_download(checksum_source, downloaded_checksum)

published_sha256 = downloaded_checksum.read_text(encoding="utf-8").split()[0].lower()
actual_sha256 = sha256_file(downloaded_archive)
assert actual_sha256 == published_sha256, "Published SHA-256 does not match the downloaded archive"
if not using_local_fallback:
    algorithm, zenodo_digest = remote_files[ARCHIVE_NAME]["checksum"].split(":", 1)
    digest = hashlib.new(algorithm)
    with downloaded_archive.open("rb") as handle:
        while block := handle.read(1 << 20):
            digest.update(block)
    assert digest.hexdigest() == zenodo_digest, "Zenodo checksum does not match"
print("Archive SHA-256:", actual_sha256)
print("Archive bytes:", downloaded_archive.stat().st_size)


Archive SHA-256: 82dfaca73e4b461294526f3453ec2c2ba2e6f8f371334712b09d2370bcab7755
Archive bytes: 20445559


In [4]:
with zipfile.ZipFile(downloaded_archive) as bundle:
    names = bundle.namelist()
    assert len(names) == len(set(names)), "Archive contains duplicate paths"
    for name in names:
        path = PurePosixPath(name)
        assert name and not path.is_absolute() and ".." not in path.parts, f"Unsafe archive path: {name}"
        assert path.parts[0] in {"data", "static", "BUNDLE_MANIFEST.json"}, f"Unexpected archive path: {name}"
    bundle_manifest = json.loads(bundle.read("BUNDLE_MANIFEST.json"))
    expected = {entry["path"]: entry for entry in bundle_manifest["files"]}
    assert set(names) == set(expected) | {"BUNDLE_MANIFEST.json"}
    for name, entry in expected.items():
        payload = bundle.read(name)
        assert len(payload) == entry["bytes"]
        assert hashlib.sha256(payload).hexdigest() == entry["sha256"]
print("Verified", len(expected), "files in the internal bundle manifest")


Verified 50 files in the internal bundle manifest


## 3. Extract atomically and validate panel tables


In [5]:
installation = Path(tempfile.mkdtemp(prefix=".linkd-install-", dir=ROOT))
try:
    with zipfile.ZipFile(downloaded_archive) as bundle:
        for member in bundle.infolist():
            if member.filename == "BUNDLE_MANIFEST.json":
                continue
            destination = installation / member.filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

    panel_manifest = json.loads((installation / "data" / "manifest.json").read_text(encoding="utf-8"))
    required_columns = {"path", "sha256", "bytes", "rows", "columns", "origin", "transformation", "panels", "privacy"}
    required_panels = {"fig1b", "fig1c", "fig2a", "fig2b", "fig2cd", "fig2e", "fig2f", "fig2g", "fig3b", "fig3c", "fig3pairs", "fig3f", "fig3g", "fig3h_edges", "fig4b_nodes", "fig4b_edges", "fig5a", "fig5d", "fig5ef", "fig5g", "fig5hi", "fig5j", "fig5k", "figs2", "figs3s4", "figs5ab", "figs5cd"}
    assert required_panels == {Path(entry["path"]).stem for entry in panel_manifest}
    for entry in panel_manifest:
        assert required_columns <= set(entry)
        path = installation / "data" / entry["path"]
        assert path.is_file() and path.stat().st_size == entry["bytes"]
        assert sha256_file(path) == entry["sha256"]
        frame = pd.read_csv(path)
        assert len(frame) == entry["rows"]
        assert list(frame.columns) == entry["columns"]

    for folder_name in ("data", "static"):
        source = installation / folder_name
        target = ROOT / folder_name
        backup = ROOT / f".{folder_name}.previous"
        if backup.exists():
            shutil.rmtree(backup)
        if target.exists():
            target.replace(backup)
        source.replace(target)
        if backup.exists():
            shutil.rmtree(backup)
finally:
    shutil.rmtree(installation, ignore_errors=True)
    shutil.rmtree(temporary_download, ignore_errors=True)

summary = pd.DataFrame(panel_manifest)[["path", "rows", "panels", "origin", "privacy"]]
display(summary)
print("Installed panel data:", ROOT / "data")
print("Installed static assets:", ROOT / "static")


,path,rows,panels,origin,privacy
0,fig1b.csv,39,fig1b,author analysis source and submitted manuscrip...,aggregate/non-identifiable
1,fig1c.csv,18,fig1c,author analysis source and submitted manuscrip...,aggregate/non-identifiable
2,fig2a.csv,14981,fig2a,author analysis source and submitted manuscrip...,aggregate/non-identifiable
3,fig2b.csv,60,fig2b,author analysis source and submitted manuscrip...,aggregate/non-identifiable
4,fig2cd.csv,18,fig2cd,author analysis source and submitted manuscrip...,aggregate/non-identifiable
5,fig2e.csv,10,fig2e,author analysis source and submitted manuscrip...,aggregate/non-identifiable
6,fig2f.csv,55534,fig2f,author analysis source and submitted manuscrip...,aggregate/non-identifiable
7,fig2g.csv,57,fig2g,author analysis source and submitted manuscrip...,aggregate/non-identifiable
8,fig3b.csv,13,fig3b,author analysis source and submitted manuscrip...,aggregate/non-identifiable
9,fig3c.csv,10014,fig3c,author analysis source and submitted manuscrip...,aggregate/non-identifiable


Installed panel data: /Users/cheng.wang/Documents/LinkD_Agent/For_Reviewer/data
Installed static assets: /Users/cheng.wang/Documents/LinkD_Agent/For_Reviewer/static


## Ready

Open any `Figure*.ipynb` notebook and choose **Restart Kernel and Run All**. Each figure notebook independently verifies the exact CSV files it reads.
